In [25]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.stats import pearsonr, ttest_1samp
from statsmodels.stats.multitest import multipletests

# Dataset directory
DATASET_DIR = Path("/Users/nedamohseni/Downloads/SoLo_dataset")

# Mobility cross-modal correlations

In [40]:
label_cols = [  "feel_lonely", "feel_isolated", "feel_connected"]

all_daily_ema = []

for participant in sorted(os.listdir(DATASET_DIR)):

    if not participant.startswith("participant_"):
        continue

    ema_path = os.path.join( DATASET_DIR, participant,  "Self_Report",  "ema_daily.csv")

    if not os.path.exists(ema_path):
        print(f"No EMA file: {participant}")
        continue

    df = pd.read_csv(ema_path)

    # Convert Unix timestamp (ms) -> America/Los_Angeles local time
    df["datetime_la"] = (pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles"))

    # Local calendar date
    df["date"] = df["datetime_la"].dt.date

    # Daily mean of the three EMA labels
    daily = (df.groupby("date", as_index=False)[label_cols].mean())

    daily.insert(0, "participant", participant)

    all_daily_ema.append(daily)

daily_ema = pd.concat(all_daily_ema, ignore_index=True)

print("Participants:", daily_ema["participant"].nunique())
print("Daily rows:", len(daily_ema))

display(daily_ema)

No EMA file: participant_groups.csv
Participants: 31
Daily rows: 3777


,participant,date,feel_lonely,feel_isolated,feel_connected
0,participant_1,2022-01-24,34.333333,30.666667,50.333333
1,participant_1,2022-01-25,0.000000,0.000000,50.000000
2,participant_1,2022-01-26,0.666667,0.000000,98.333333
3,participant_1,2022-01-27,1.000000,0.000000,99.000000
4,participant_1,2022-01-28,1.000000,0.000000,66.000000
...,...,...,...,...,...
3772,participant_9,2022-06-25,18.000000,9.333333,71.000000
3773,participant_9,2022-06-26,35.333333,15.000000,64.333333
3774,participant_9,2022-06-27,23.000000,12.333333,80.666667
3775,participant_9,2022-06-28,69.000000,19.000000,33.000000


In [41]:
all_mobility = []
for participant in sorted(os.listdir(DATASET_DIR)):

    if not participant.startswith("participant_"):
        continue

    mobility_path = os.path.join( DATASET_DIR, participant, "AWARE","mobility_features.csv")

    if not os.path.exists(mobility_path):
        print(f"No mobility file: {participant}")
        continue

    df = pd.read_csv(mobility_path)

    df["date"] = pd.to_datetime(df["date"]).dt.date

    all_mobility.append(df)

daily_mobility = pd.concat(all_mobility, ignore_index=True)

print("Participants:", daily_mobility["participant"].nunique())
print("Daily mobility rows:", len(daily_mobility))

display(daily_mobility)

No mobility file: participant_18
No mobility file: participant_27
No mobility file: participant_groups.csv
Participants: 29
Daily mobility rows: 2740


,participant,date,locationvariance,loglocationvariance,totaldistance,avgspeed,varspeed,numberofsignificantplaces,numberlocationtransitions,radiusgyration,...,outlierstimepercent,maxlengthstayatclusters,minlengthstayatclusters,avglengthstayatclusters,stdlengthstayatclusters,locationentropy,normalizedlocationentropy,timeathome,homelabel,minutesdataused
0,participant_1,2021-11-12,7.106724e-06,-5.148331,3098.653878,4.082839,0.771587,1.0,0.0,0.000000,...,0.0,28.470433,28.470433,28.470433,0.000000,0.000000,0.000000,0.000000,1.0,74.007183
1,participant_1,2021-11-13,9.183273e-06,-5.037002,1224.087093,1.906240,22.471702,1.0,0.0,0.000000,...,0.0,29.492517,29.492517,29.492517,0.000000,0.000000,0.000000,17.489150,1.0,68.021367
2,participant_1,2021-11-14,6.354017e-07,-6.196952,48.166381,23.920399,245.344094,1.0,0.0,0.000000,...,0.0,77.395100,77.395100,77.395100,0.000000,0.000000,0.000000,49.142183,1.0,77.515917
3,participant_1,2021-11-15,4.513150e-06,-5.345520,655.108885,1.582665,0.276840,1.0,0.0,0.000000,...,0.0,29.433450,29.433450,29.433450,0.000000,0.000000,0.000000,19.739117,1.0,54.269117
4,participant_1,2021-11-16,9.623346e-06,-5.016674,4656.349609,4.627509,16.689975,3.0,2.0,280.736256,...,0.0,21.790350,1.297767,14.896161,11.776938,0.804399,0.268133,21.790350,1.0,105.062433
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2735,participant_9,2022-06-06,2.030151e-28,-27.692472,0.000000,0.000000,0.000000,1.0,0.0,0.000000,...,0.0,3.172100,3.172100,3.172100,0.000000,0.000000,0.000000,0.000000,1.0,3.172100
2736,participant_9,2022-06-07,0.000000e+00,NaN,0.000000,0.000000,0.000000,1.0,0.0,0.000000,...,0.0,5.311283,5.311283,5.311283,0.000000,0.000000,0.000000,0.000000,1.0,5.311283
2737,participant_9,2022-06-09,0.000000e+00,NaN,105.378940,2.181621,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.898183
2738,participant_9,2022-06-15,0.000000e+00,NaN,0.000000,0.000000,0.000000,1.0,0.0,0.000000,...,0.0,2.016833,2.016833,2.016833,0.000000,0.000000,0.000000,0.000000,1.0,2.016833


In [42]:
merged = daily_mobility.merge(daily_ema, on=["participant", "date"], how="inner")

print("Merged rows:", len(merged))
print("Participants:", merged["participant"].nunique())
display(merged.head())

Merged rows: 1862
Participants: 29


,participant,date,locationvariance,loglocationvariance,totaldistance,avgspeed,varspeed,numberofsignificantplaces,numberlocationtransitions,radiusgyration,...,avglengthstayatclusters,stdlengthstayatclusters,locationentropy,normalizedlocationentropy,timeathome,homelabel,minutesdataused,feel_lonely,feel_isolated,feel_connected
0,participant_1,2022-01-30,3.581604e-07,-6.445922,250.751521,1.827069,0.069280,1.0,0.0,0.0,...,89.391800,0.0,0.0,0.0,89.391800,1.0,97.626350,0.666667,1.333333,32.0
1,participant_1,2022-02-02,4.284296e-06,-5.368121,1092.894346,4.464509,1.436016,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,14.687767,0.000000,3.000000,97.0
2,participant_1,2022-02-11,2.077151e-28,-27.682532,16.534166,1.652499,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.600333,0.000000,0.000000,100.0
3,participant_1,2022-02-12,0.000000e+00,NaN,0.000000,0.000000,0.000000,1.0,0.0,0.0,...,19.506433,0.0,0.0,0.0,19.506433,1.0,19.506433,24.333333,20.666667,0.0
4,participant_1,2022-02-16,0.000000e+00,NaN,30.642937,12.342199,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.148967,0.000000,1.000000,50.0


In [43]:
mobility_features = ["locationvariance", "totaldistance", "varspeed", "radiusgyration", "timeattop1location", "avglengthstayatclusters"]

participant_corrs = []
for feature in mobility_features:
    for label in label_cols:
        for participant, df_p in merged.groupby("participant"):
            pair = df_p[[feature, label]].dropna()
            if len(pair) < 3 or pair[feature].nunique() < 2 or pair[label].nunique() < 2:
                continue
            r, _ = pearsonr(pair[feature], pair[label])
            z = np.arctanh(np.clip(r, -0.999999, 0.999999))
            participant_corrs.append({"participant": participant, "mobility_feature": feature, "ema_label": label, "n_days": len(pair), "r": r, "z": z})

participant_corrs = pd.DataFrame(participant_corrs)

group_results = []

for (feature, label), df_pair in participant_corrs.groupby(["mobility_feature", "ema_label"]):
    z_values = df_pair["z"].dropna().values
    if len(z_values) < 2:
        continue
    mean_z = np.mean(z_values)
    mean_r = np.tanh(mean_z)
    t_stat, p_raw = ttest_1samp(z_values, popmean=0, alternative="two-sided")
    group_results.append({"mobility_feature": feature, "ema_label": label, "n_participants": len(z_values), "mean_r": mean_r, "mean_z": mean_z, "t_stat": t_stat, "p_raw": p_raw})

mobility_results = pd.DataFrame(group_results)
mobility_results["p_fdr"] = multipletests(mobility_results["p_raw"].values, alpha=0.05, method="fdr_bh")[1]
mobility_results["sig_fdr"] = mobility_results["p_fdr"].apply(lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")
mobility_results["table_value"] = mobility_results.apply(lambda row: f"{row['mean_r']:.2f}{row['sig_fdr']} (N={int(row['n_participants'])})", axis=1)

final_table = mobility_results.pivot(index="mobility_feature", columns="ema_label", values="table_value")
final_table = final_table.reindex(mobility_features)
final_table = final_table[["feel_lonely", "feel_isolated", "feel_connected"]]

final_table


ema_label,feel_lonely,feel_isolated,feel_connected
mobility_feature,,,
locationvariance,-0.10** (N=28),-0.08* (N=29),0.07 (N=29)
totaldistance,-0.08* (N=28),-0.06 (N=29),0.10* (N=29)
varspeed,-0.12*** (N=28),-0.09* (N=29),0.10* (N=29)
radiusgyration,-0.09* (N=27),-0.07* (N=28),0.08 (N=28)
timeattop1location,0.07 (N=27),0.14 (N=28),-0.11* (N=28)
avglengthstayatclusters,0.09* (N=27),0.18 (N=28),-0.14** (N=28)


# Oura sleep cross-modal correlations

In [86]:
all_morning_ema = []

for participant in sorted(os.listdir(DATASET_DIR)):

    if not participant.startswith("participant_"):
        continue

    ema_path = DATASET_DIR / participant / "Self_Report" / "ema_daily.csv"

    if not ema_path.exists():
        continue

    df = pd.read_csv(ema_path)

    # morning EMA only
    df = df[df["surveyindex"] == 0].copy()

    # Convert survey timestamp to Pacific time
    df["datetime_la"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")
    df["date"] = df["datetime_la"].dt.date

    all_morning_ema.append(df)

morning_ema = pd.concat(all_morning_ema, ignore_index=True)

print("Participants:", morning_ema["participant"].nunique())
print("Morning EMA records:", len(morning_ema))

display(morning_ema[["participant", "date", "timestamp", "datetime_la", "sleep_time", "wake_time", "rested_score", "quality_score"]].head(10))

Participants: 31
Morning EMA records: 3306


,participant,date,timestamp,datetime_la,sleep_time,wake_time,rested_score,quality_score
0,participant_1,2022-01-24,1643053391977,2022-01-24 11:43:11.977000-08:00,1.643019e+12,1.643052e+12,98.0,80.0
1,participant_1,2022-01-25,1643133350138,2022-01-25 09:55:50.138000-08:00,1.643143e+12,1.643130e+12,80.0,70.0
2,participant_1,2022-01-26,1643221876114,2022-01-26 10:31:16.114000-08:00,1.643238e+12,1.643221e+12,95.0,93.0
3,participant_1,2022-01-27,1643312718026,2022-01-27 11:45:18.026000-08:00,1.643325e+12,1.643302e+12,96.0,100.0
4,participant_1,2022-01-28,1643394667250,2022-01-28 10:31:07.250000-08:00,1.643406e+12,1.643391e+12,89.0,23.0
5,participant_1,2022-01-29,1643481436290,2022-01-29 10:37:16.290000-08:00,1.643489e+12,1.643481e+12,19.0,20.0
6,participant_1,2022-01-30,1643565342260,2022-01-30 09:55:42.260000-08:00,1.643587e+12,1.643565e+12,95.0,96.0
7,participant_1,2022-01-31,1643648668875,2022-01-31 09:04:28.875000-08:00,1.643665e+12,1.643649e+12,96.0,95.0
8,participant_1,2022-02-01,1643733076319,2022-02-01 08:31:16.319000-08:00,1.643755e+12,1.643733e+12,77.0,64.0
9,participant_1,2022-02-03,1643913273513,2022-02-03 10:34:33.513000-08:00,1.643927e+12,1.643912e+12,62.0,40.0


In [87]:
all_oura = []

for participant in sorted(os.listdir(DATASET_DIR)):

    if not participant.startswith("participant_"):
        continue

    oura_path = DATASET_DIR / participant / "Oura" / "sleep_daily.csv"

    if not oura_path.exists():
        print(f"No Oura file: {participant}")
        continue

    df = pd.read_csv(oura_path)

    # Unix timestamps (ms) to Pacific time conversion
    df["oura_bedtime_la"] = pd.to_datetime(df["sleep_bedtime_start"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")
    df["oura_wake_la"] = pd.to_datetime(df["sleep_bedtime_end"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")

    # Extract actual wake-up date 
    df["wake_date"] = df["oura_wake_la"].dt.date

    all_oura.append(df)

daily_oura = pd.concat(all_oura, ignore_index=True)

print("Participants:", daily_oura["participant"].nunique())
print("Oura sleep records:", len(daily_oura))

display(daily_oura[["participant", "date", "oura_bedtime_la", "oura_wake_la", "wake_date", "sleep_duration", "sleep_total", "sleep_score"]].head(10))

No Oura file: participant_groups.csv
Participants: 31
Oura sleep records: 4283


,participant,date,oura_bedtime_la,oura_wake_la,wake_date,sleep_duration,sleep_total,sleep_score
0,participant_1,2022-01-24,2022-01-25 01:06:51-08:00,2022-01-25 09:15:51-08:00,2022-01-25,29340.0,26790.0,74.0
1,participant_1,2022-01-25,2022-01-26 01:43:02-08:00,2022-01-26 09:53:02-08:00,2022-01-26,29400.0,27480.0,77.0
2,participant_1,2022-01-26,2022-01-27 02:25:59-08:00,2022-01-27 08:37:59-08:00,2022-01-27,22320.0,19320.0,55.0
3,participant_1,2022-01-27,2022-01-28 01:37:01-08:00,2022-01-28 08:54:01-08:00,2022-01-28,26220.0,23970.0,69.0
4,participant_1,2022-01-28,2022-01-29 05:33:32-08:00,2022-01-29 10:36:32-08:00,2022-01-29,18180.0,12450.0,39.0
5,participant_1,2022-01-29,2022-01-30 03:51:57-08:00,2022-01-30 09:53:57-08:00,2022-01-30,21720.0,20640.0,68.0
6,participant_1,2022-01-30,2022-01-31 02:36:32-08:00,2022-01-31 10:32:32-08:00,2022-01-31,28560.0,25680.0,72.0
7,participant_1,2022-01-31,2022-02-01 02:07:33-08:00,2022-02-01 11:15:33-08:00,2022-02-01,32880.0,27930.0,73.0
8,participant_1,2022-02-01,2022-02-02 03:56:29-08:00,2022-02-02 11:35:29-08:00,2022-02-02,27540.0,25770.0,70.0
9,participant_1,2022-02-02,2022-02-03 02:01:59-08:00,2022-02-03 09:39:59-08:00,2022-02-03,27480.0,21330.0,58.0


In [77]:
# Keep longest Oura sleep session per participant-date
daily_oura = daily_oura.sort_values("sleep_duration", ascending=False).drop_duplicates(["participant", "wake_date"])

# Keep earliest morning EMA submission per participant-date
morning_ema = morning_ema.sort_values("timestamp").drop_duplicates(["participant", "date"])

# Merge Oura with morning EMA
merged_sleep = daily_oura.merge(morning_ema, 
                                left_on=["participant", "wake_date"], 
                                right_on=["participant", "date"],
                                how="inner", 
                                suffixes=("_oura", "_ema"), 
                                validate="one_to_one")

print("Matched records:", len(merged_sleep))
print("Participants:", merged_sleep["participant"].nunique())
print("Duplicate matches:", merged_sleep.duplicated(["participant", "wake_date"]).sum())

display(merged_sleep[["participant", "wake_date", "sleep_score", "quality_score", "rested_score", "sleep_duration", "sleep_total"]].head(10))

Matched records: 2792
Participants: 31
Duplicate matches: 0


,participant,wake_date,sleep_score,quality_score,rested_score,sleep_duration,sleep_total
0,participant_11,2022-04-08,83.0,52.0,74.0,54000.0,40140.0
1,participant_17,2022-02-05,81.0,39.0,38.0,54000.0,40770.0
2,participant_12,2022-02-14,68.0,64.0,66.0,54000.0,25140.0
3,participant_15,2022-05-15,63.0,31.0,40.0,54000.0,32250.0
4,participant_15,2022-03-01,78.0,27.0,67.0,54000.0,42420.0
5,participant_15,2022-01-25,51.0,19.0,20.0,54000.0,28440.0
6,participant_15,2022-04-05,78.0,36.0,79.0,54000.0,37710.0
7,participant_21,2022-03-18,78.0,50.0,41.0,53160.0,36900.0
8,participant_15,2022-05-03,72.0,45.0,61.0,52980.0,36570.0
9,participant_17,2022-05-07,81.0,67.0,64.0,52500.0,39420.0


In [88]:
sleep_pairs = [("sleep_score", "quality_score"), ("sleep_score", "rested_score")]

participant_sleep_corrs = []

for oura_var, ema_var in sleep_pairs:
    for participant, df_p in merged_sleep.groupby("participant"):
        pair = df_p[[oura_var, ema_var]].dropna()

        if len(pair) < 3 or pair[oura_var].nunique() < 2 or pair[ema_var].nunique() < 2:
            continue

        r, _ = pearsonr(pair[oura_var], pair[ema_var])
        z = np.arctanh(np.clip(r, -0.999999, 0.999999))

        participant_sleep_corrs.append({
            "participant": participant,
            "oura_var": oura_var,
            "ema_var": ema_var,
            "n_days": len(pair),
            "r": r,
            "z": z
        })

participant_sleep_corrs = pd.DataFrame(participant_sleep_corrs)

group_results = []

for (oura_var, ema_var), df_pair in participant_sleep_corrs.groupby(["oura_var", "ema_var"]):
    z_values = df_pair["z"].dropna().values

    if len(z_values) < 2:
        continue

    mean_z = np.mean(z_values)
    mean_r = np.tanh(mean_z)
    t_stat, p_raw = ttest_1samp(z_values, popmean=0, alternative="two-sided")

    group_results.append({
        "oura_var": oura_var,
        "ema_var": ema_var,
        "n_participants": len(z_values),
        "mean_r": mean_r,
        "mean_z": mean_z,
        "t_stat": t_stat,
        "p_raw": p_raw
    })

sleep_results = pd.DataFrame(group_results)

sleep_results["p_fdr"] = multipletests(sleep_results["p_raw"], method="fdr_bh")[1]

display(sleep_results)

,oura_var,ema_var,n_participants,mean_r,mean_z,t_stat,p_raw,p_fdr
0,sleep_score,quality_score,31,0.233766,0.238170,3.397509,0.001936,0.001936
1,sleep_score,rested_score,31,0.314919,0.325996,6.064731,0.000001,0.000002


In [96]:
# Convert EMA sleep/wake timestamps to Pacific time
merged_sleep["ema_sleep_la"] = pd.to_datetime(merged_sleep["sleep_time"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")
merged_sleep["ema_wake_la"] = pd.to_datetime(merged_sleep["wake_time"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")

# Calculate EMA sleep duration
merged_sleep["ema_sleep_duration_hours"] = (merged_sleep["ema_wake_la"] - merged_sleep["ema_sleep_la"]).dt.total_seconds() / 3600

# Keep plausible EMA durations and convert Oura duration to hours
sleep_duration_df = merged_sleep[merged_sleep["ema_sleep_duration_hours"].between(3, 16)].copy()
sleep_duration_df["oura_sleep_duration_hours"] = sleep_duration_df["sleep_duration"] / 3600

print("Valid matched records:", len(sleep_duration_df))
print("Participants:", sleep_duration_df["participant"].nunique())

Valid matched records: 2163
Participants: 30


In [97]:
duration_corrs = []

for participant, df_p in sleep_duration_df.groupby("participant"):
    pair = df_p[["ema_sleep_duration_hours", "oura_sleep_duration_hours"]].dropna()

    if len(pair) < 3 or pair["ema_sleep_duration_hours"].nunique() < 2 or pair["oura_sleep_duration_hours"].nunique() < 2:
        continue

    r, _ = pearsonr(pair["ema_sleep_duration_hours"], pair["oura_sleep_duration_hours"])
    z = np.arctanh(np.clip(r, -0.999999, 0.999999))

    duration_corrs.append({"participant": participant, "n_days": len(pair), "r": r, "z": z})

duration_corrs = pd.DataFrame(duration_corrs)

z_values = duration_corrs["z"].values
mean_r = np.tanh(z_values.mean())
t_stat, p_raw = ttest_1samp(z_values, popmean=0)

print("Participants:", len(z_values))
print("Mean correlation:", round(mean_r, 3))
print("P-value:", p_raw)

Participants: 29
Mean correlation: 0.812
P-value: 8.650236411814639e-15


In [98]:
# Combine all three correlation results
duration_result = pd.DataFrame([{"oura_var": "oura_sleep_duration_hours", "ema_var": "ema_sleep_duration_hours", "n_participants": len(z_values), "mean_r": mean_r, "mean_z": z_values.mean(), "t_stat": t_stat, "p_raw": p_raw}])

final_sleep_results = pd.concat([sleep_results.drop(columns="p_fdr"), duration_result], ignore_index=True)

# Apply BH-FDR correction across all three tests
final_sleep_results["p_fdr"] = multipletests(final_sleep_results["p_raw"], method="fdr_bh")[1]

display(final_sleep_results)

,oura_var,ema_var,n_participants,mean_r,mean_z,t_stat,p_raw,p_fdr
0,sleep_score,quality_score,31,0.233766,0.238170,3.397509,1.936242e-03,1.936242e-03
1,sleep_score,rested_score,31,0.314919,0.325996,6.064731,1.163665e-06,1.745497e-06
2,oura_sleep_duration_hours,ema_sleep_duration_hours,29,0.811552,1.131557,14.831760,8.650236e-15,2.595071e-14
